# grok-007 · Merge 发布 + 终评 + 模型卡（学习向）

> **你在学什么**：训练产物如何变成 **可交付权重**：
> 1. LoRA 训练  
> 2. `merge_and_unload` 合并进基座  
> 3. 终评 + MODEL_CARD（数据、指标、用法、局限）  
>
> **关键概念**：adapter 小而灵活；merged 部署简单；发布必须写清 **适用/不适用**。


# grok-007-release-merge-card

**M3 Release:** train LoRA, **merge**, final eval vs base, write MODEL_CARD + final_report.

Outputs: `mini_instruct_t4_merged/`, `MODEL_CARD.md`, `final_report.json`, `grok007_results.json`


In [ ]:
# 【步骤】锁定单卡：只让进程看见 GPU0（学习「逻辑单卡」）
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')


In [ ]:
# 【步骤】检查有几张 GPU、名字是否为 Tesla T4
import os, re, json, time, random, hashlib, platform, subprocess, sys
from pathlib import Path
from typing import Any, Optional

import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

OUT = Path('/kaggle/working')
DATA = OUT / 'data'
EVAL = OUT / 'eval_suite_v1'
for p in [OUT, DATA, EVAL]:
    p.mkdir(parents=True, exist_ok=True)

print('python', platform.python_version())
print('torch', torch.__version__)
assert torch.cuda.is_available()
assert torch.cuda.device_count() == 1, torch.cuda.device_count()
print('device', torch.cuda.get_device_name(0))
DEVICE = torch.device('cuda:0')  # 主设备：默认第一张可见 GPU
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

AMP=torch.float16  # T4 无 fp16（无原生 bf16 tensor core）


In [ ]:
# 【步骤】代码题：用子进程执行+assert，能量化「能不能跑」

def uid(*parts) -> str:
    return hashlib.sha1('||'.join(map(str, parts)).encode()).hexdigest()[:16]

def normalize_text(s: str) -> str:
    s = (s or '').strip().lower()
    s = re.sub(r'\s+', ' ', s)
    return s.replace('，', ',').replace('。', '.').replace('：', ':')

def extract_final_number(text: str):
    if text is None:
        return None
    m = re.findall(r'-?\d+(?:\.\d+)?', text.replace(',', ''))
    return m[-1] if m else None

def safe_json_loads(text: str):
    if not text:
        return None
    t = text.strip()
    t = re.sub(r'^```(?:json)?\s*', '', t)
    t = re.sub(r'\s*```$', '', t)
    try:
        return json.loads(t)
    except Exception:
        pass
    m = re.search(r'\{[\s\S]*\}', t)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return None
    return None

def run_python_user_code(code: str, tests: str, timeout_s: float = 2.0) -> bool:
    prog = code + '\n\n' + tests + '\nprint("OK")\n'
    try:
        r = subprocess.run(
            [sys.executable, '-c', prog],
            capture_output=True, text=True, timeout=timeout_s,
            env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1'},
        )
        return r.returncode == 0 and 'OK' in (r.stdout or '')
    except Exception:
        return False


In [ ]:
# 【步骤】构造数学 train/eval（eval 用不同随机流，降低泄漏）

def make_math_train(n=2000):
    rows = []
    rng = random.Random(SEED)
    import math as _m
    while len(rows) < n:
        kind = rng.choice(['mul', 'add', 'sub', 'pct', 'avg', 'seq', 'gcd', 'area'])
        if kind == 'mul':
            a, b = rng.randint(3, 40), rng.randint(3, 40)
            q, ans = f'What is {a} * {b}?', str(a * b)
        elif kind == 'add':
            a, b = rng.randint(10, 200), rng.randint(10, 200)
            q, ans = f'Compute {a} + {b}.', str(a + b)
        elif kind == 'sub':
            a, b = rng.randint(50, 300), rng.randint(1, 49)
            q, ans = f'What is {a} - {b}?', str(a - b)
        elif kind == 'pct':
            price, off = rng.choice([80, 100, 120, 200, 250]), rng.choice([10, 20, 25, 50])
            q = f'A ${price} item is {off}% off. What is the sale price?'
            ans = str(int(price * (100 - off) / 100))
        elif kind == 'avg':
            xs = [rng.randint(1, 20) for _ in range(3)]
            q = f'Average of {xs[0]}, {xs[1]}, {xs[2]}?'
            ans = str(sum(xs) // 3) if sum(xs) % 3 == 0 else str(round(sum(xs) / 3, 2))
        elif kind == 'seq':
            a, r = rng.randint(2, 5), rng.randint(2, 3)
            seq = [a * (r ** i) for i in range(4)]
            q = f'What is the next number in the sequence {", ".join(map(str, seq))}?'
            ans = str(a * (r ** 4))
        elif kind == 'gcd':
            a, b = rng.randint(12, 90), rng.randint(12, 90)
            q, ans = f'GCD of {a} and {b}?', str(_m.gcd(a, b))
        else:
            w, h = rng.randint(3, 20), rng.randint(3, 20)
            q, ans = f'Area of a {w} by {h} rectangle?', str(w * h)
        rows.append({
            'id': uid('math_train', q, ans),
            'task_type': 'math',
            'instruction': q + ' Reply with the final number only.',
            'output': ans,
            'meta': {'kind': kind},
        })
    seen, out = set(), []
    for r in rows:
        if r['id'] not in seen:
            seen.add(r['id']); out.append(r)
    return out[:n]


def make_math_eval(n=120):
    rng = random.Random(SEED + 999)
    rows = []
    while len(rows) < n:
        a, b = rng.randint(11, 49), rng.randint(11, 49)
        kind = rng.choice(['mul', 'add', 'sub', 'pct', 'area'])
        if kind == 'mul':
            q, ans = f'What is {a} * {b}?', str(a * b)
        elif kind == 'add':
            q, ans = f'Compute {a} + {b}.', str(a + b)
        elif kind == 'sub':
            a = max(a, b) + rng.randint(5, 30)
            q, ans = f'What is {a} - {b}?', str(a - b)
        elif kind == 'pct':
            price, off = rng.choice([60, 90, 150, 180, 240]), rng.choice([10, 15, 20, 25, 30])
            q = f'An item costs ${price}. Discount {off}%. Sale price?'
            ans = str(int(price * (100 - off) / 100))
        else:
            w, h = rng.randint(4, 25), rng.randint(4, 25)
            q, ans = f'Rectangle {w} x {h}. Area?', str(w * h)
        rows.append({
            'id': uid('math_eval', q, ans),
            'task_type': 'math_em',
            'prompt': q + ' Reply with the final number only.',
            'gold': ans,
            'meta': {'kind': kind},
        })
    seen, out = set(), []
    for r in rows:
        if r['id'] not in seen:
            seen.add(r['id']); out.append(r)
    return out[:n]

math_train = make_math_train(2200)
math_eval = make_math_eval(120)
print('math', len(math_train), len(math_eval))


In [ ]:
# 【步骤】代码题：用子进程执行+assert，能量化「能不能跑」

CODE_TRAIN_TEMPLATES = [
    ('Write a Python function add(a, b) that returns the sum of a and b.',
     'def add(a, b):\n    return a + b\n',
     'assert add(2,3)==5\nassert add(-1,1)==0\n'),
    ('Write a Python function is_palindrome(s) that returns True if s equals its reverse.',
     'def is_palindrome(s):\n    return s == s[::-1]\n',
     'assert is_palindrome("aba")\nassert not is_palindrome("ab")\n'),
    ('Write a Python function factorial(n) for n>=0.',
     'def factorial(n):\n    r=1\n    for i in range(2,n+1):\n        r*=i\n    return r\n',
     'assert factorial(0)==1\nassert factorial(5)==120\n'),
    ('Write a Python function max3(a,b,c) returning the maximum of three numbers.',
     'def max3(a,b,c):\n    return max(a,b,c)\n',
     'assert max3(1,5,3)==5\nassert max3(9,9,1)==9\n'),
    ('Write a Python function count_vowels(s) counting aeiou case-insensitive.',
     'def count_vowels(s):\n    return sum(1 for ch in s.lower() if ch in "aeiou")\n',
     'assert count_vowels("Hello")==2\nassert count_vowels("xyz")==0\n'),
    ('Write a Python function reverse_list(xs) returning a new reversed list.',
     'def reverse_list(xs):\n    return xs[::-1]\n',
     'assert reverse_list([1,2,3])==[3,2,1]\n'),
    ('Write a Python function is_even(n) returning True if n is even.',
     'def is_even(n):\n    return n % 2 == 0\n',
     'assert is_even(4)\nassert not is_even(7)\n'),
    ('Write a Python function clamp(x, lo, hi) clamping x into [lo, hi].',
     'def clamp(x, lo, hi):\n    return max(lo, min(hi, x))\n',
     'assert clamp(5,0,10)==5\nassert clamp(-1,0,10)==0\nassert clamp(99,0,10)==10\n'),
    ('Write a Python function word_count(s) returning whitespace-separated word count.',
     'def word_count(s):\n    return len(s.split())\n',
     'assert word_count("a b c")==3\nassert word_count("")==0\n'),
    ('Write a Python function unique_sorted(xs) returning sorted unique elements.',
     'def unique_sorted(xs):\n    return sorted(set(xs))\n',
     'assert unique_sorted([3,1,2,1])==[1,2,3]\n'),
]

def make_code_train():
    rows = []
    for instr, sol, tests in CODE_TRAIN_TEMPLATES:
        assert run_python_user_code(sol, tests), instr
        rows.append({
            'id': uid('code_train', instr),
            'task_type': 'code',
            'instruction': instr + ' Output only the Python code, no explanation.',
            'output': sol.strip() + '\n',
            'meta': {'tests': tests},
        })
    extra = []
    for r in rows:
        extra.append({**r, 'id': uid('code_train2', r['instruction']), 'instruction': 'Task: ' + r['instruction']})
    return rows + extra

CODE_EVAL = [
    ('Write a Python function mul(a, b) that returns the product of a and b.',
     'def mul(a, b):\n    return a * b\n', 'assert mul(3,4)==12\nassert mul(0,9)==0\n'),
    ('Write a Python function is_anagram(a, b) True if a and b have same characters.',
     'def is_anagram(a, b):\n    return sorted(a)==sorted(b)\n', 'assert is_anagram("ab","ba")\nassert not is_anagram("ab","cd")\n'),
    ('Write a Python function sum_list(xs) returning the sum of numbers in xs.',
     'def sum_list(xs):\n    return sum(xs)\n', 'assert sum_list([1,2,3])==6\nassert sum_list([])==0\n'),
    ('Write a Python function fizz(n) returning "fizz" if n divisible by 3 else str(n).',
     'def fizz(n):\n    return "fizz" if n%3==0 else str(n)\n', 'assert fizz(3)=="fizz"\nassert fizz(4)=="4"\n'),
    ('Write a Python function last_char(s) returning the last character of non-empty s.',
     'def last_char(s):\n    return s[-1]\n', 'assert last_char("cat")=="t"\n'),
    ('Write a Python function mean(xs) returning mean of non-empty numeric list xs.',
     'def mean(xs):\n    return sum(xs)/len(xs)\n', 'assert abs(mean([2,4])-3)<1e-9\n'),
    ('Write a Python function starts_with_a(s) True if s starts with a/A.',
     'def starts_with_a(s):\n    return len(s)>0 and s[0].lower()=="a"\n', 'assert starts_with_a("Apple")\nassert not starts_with_a("banana")\n'),
    ('Write a Python function flatten2(xss) flattening one level of list-of-lists.',
     'def flatten2(xss):\n    out=[]\n    for xs in xss:\n        out.extend(xs)\n    return out\n', 'assert flatten2([[1],[2,3]])==[1,2,3]\n'),
    ('Write a Python function digit_sum(n) sum of decimal digits of non-negative int n.',
     'def digit_sum(n):\n    return sum(int(ch) for ch in str(n))\n', 'assert digit_sum(123)==6\nassert digit_sum(0)==0\n'),
    ('Write a Python function repeat_str(s, n) returning s repeated n times.',
     'def repeat_str(s, n):\n    return s * n\n', 'assert repeat_str("ab",3)=="ababab"\n'),
]

def make_code_eval():
    rows = []
    for instr, sol, tests in CODE_EVAL:
        assert run_python_user_code(sol, tests)
        rows.append({
            'id': uid('code_eval', instr),
            'task_type': 'code_exec',
            'prompt': instr + ' Output only the Python code, no explanation.',
            'gold_code': sol,
            'tests': tests,
            'meta': {},
        })
    rng = random.Random(SEED + 7)
    for i in range(50):
        name = f'fn_{i}'
        k = rng.randint(2, 9)
        instr = f'Write a Python function {name}(x) that returns x + {k}.'
        sol = f'def {name}(x):\n    return x + {k}\n'
        tests = f'assert {name}(0)=={k}\nassert {name}(3)=={3+k}\n'
        rows.append({
            'id': uid('code_eval_syn', instr),
            'task_type': 'code_exec',
            'prompt': instr + ' Output only the Python code, no explanation.',
            'gold_code': sol,
            'tests': tests,
            'meta': {'synthetic': True},
        })
    return rows

code_train = make_code_train()
code_eval = make_code_eval()
print('code', len(code_train), len(code_eval))


In [ ]:

def make_struct_train(n=900):
    rng = random.Random(SEED + 3)
    rows = []
    for i in range(n // 2):
        name = rng.choice(['Alice', 'Bob', 'Carol', 'Diego', 'Eve', 'Fang'])
        age = rng.randint(18, 60)
        city = rng.choice(['Paris', 'Berlin', 'Tokyo', 'Austin', 'Chengdu', 'Lisbon'])
        obj = {'name': name, 'age': age, 'city': city}
        rows.append({
            'id': uid('json_train', name, age, city),
            'task_type': 'json',
            'instruction': f'Return a JSON object with keys name, age, city for {name}, age {age}, city {city}. JSON only.',
            'output': json.dumps(obj, ensure_ascii=False),
            'meta': {'schema': ['name', 'age', 'city']},
        })
    for i in range(n // 2):
        table = rng.choice(['users', 'orders', 'products'])
        col = rng.choice(['active', 'status', 'deleted'])
        val = rng.choice([0, 1, '"paid"', '"open"'])
        sql = f'SELECT * FROM {table} WHERE {col} = {val};'
        rows.append({
            'id': uid('sql_train', sql),
            'task_type': 'sql',
            'instruction': f'Write SQL: all rows from {table} where {col} = {val}. One line SQL only.',
            'output': sql,
            'meta': {},
        })
    return rows

def make_struct_eval():
    rng = random.Random(SEED + 11)
    json_rows, sql_rows = [], []
    for i in range(50):
        name = rng.choice(['Gao', 'Mina', 'Omar', 'Priya', 'Sam', 'Yuki'])
        age = rng.randint(18, 70)
        city = rng.choice(['Seoul', 'Nairobi', 'Recife', 'Oslo', 'Xiamen'])
        obj = {'name': name, 'age': age, 'city': city}
        json_rows.append({
            'id': uid('json_eval', name, age, city),
            'task_type': 'json_valid',
            'prompt': f'Return JSON with keys name, age, city for person {name}, age {age}, lives in {city}. JSON only.',
            'gold': obj,
            'meta': {'keys': ['name', 'age', 'city']},
        })
    for i in range(40):
        table = rng.choice(['customers', 'invoices', 'events'])
        col = rng.choice(['active', 'paid', 'visible'])
        val = rng.choice([0, 1])
        sql = f'SELECT * FROM {table} WHERE {col} = {val};'
        sql_rows.append({
            'id': uid('sql_eval', sql),
            'task_type': 'sql_form',
            'prompt': f'Write one-line SQL selecting all columns from {table} where {col} equals {val}. SQL only.',
            'gold': sql,
            'meta': {'table': table, 'col': col, 'val': val},
        })
    return json_rows, sql_rows

ZH_PAIRS = [
    ('用一句话解释什么是梯度下降。', '梯度下降沿着损失函数梯度的反方向迭代更新参数以最小化损失。'),
    ('用一句话解释 LoRA。', 'LoRA 冻结原模型权重，仅训练注入到线性层中的低秩适配矩阵。'),
    ('PEFT 的英文全称是什么？', 'Parameter-Efficient Fine-Tuning'),
    ('把下面译成中文：Machine learning is powerful.', '机器学习很强大。'),
    ('用一句话说明混合精度训练。', '用低精度计算加速并省显存，同时用高精度保存关键权重以保证稳定。'),
    ('余弦学习率衰减有什么好处？', '它平滑降低学习率，常有助于训练后期收敛。'),
    ('什么是过拟合？一句话。', '模型在训练集上表现很好但在新数据上泛化变差。'),
    ('一句话说明什么是注意力机制。', '注意力机制根据相关性为不同输入位置分配权重以聚合信息。'),
]

def make_zh_train():
    rows = []
    for q, a in ZH_PAIRS:
        rows.append({'id': uid('zh_train', q), 'task_type': 'zh', 'instruction': q, 'output': a, 'meta': {}})
        rows.append({'id': uid('zh_train2', q), 'task_type': 'zh', 'instruction': '请简要回答：' + q, 'output': a, 'meta': {}})
    return rows

def make_zh_eval():
    items = [
        ('请用一句话解释梯度下降算法。', '梯度下降沿着损失函数梯度的反方向迭代更新参数以最小化损失。'),
        ('请用一句话解释什么是 LoRA 微调。', 'LoRA 冻结原模型权重，仅训练注入到线性层中的低秩适配矩阵。'),
        ('PEFT 代表什么？', 'Parameter-Efficient Fine-Tuning'),
        ('翻译成中文：Machine learning is powerful', '机器学习很强大。'),
        ('混合精度训练是什么？请一句话。', '用低精度计算加速并省显存，同时用高精度保存关键权重以保证稳定。'),
        ('为什么使用余弦学习率衰减？', '它平滑降低学习率，常有助于训练后期收敛。'),
        ('过拟合是什么意思？', '模型在训练集上表现很好但在新数据上泛化变差。'),
        ('注意力机制是做什么的？一句话。', '注意力机制根据相关性为不同输入位置分配权重以聚合信息。'),
    ]
    rows = []
    for q, a in items:
        rows.append({'id': uid('zh_eval', q), 'task_type': 'zh_brief', 'prompt': q, 'gold': a, 'meta': {}})
        rows.append({'id': uid('zh_eval2', q), 'task_type': 'zh_brief', 'prompt': f'简答：{q}', 'gold': a, 'meta': {}})
    extra = [('法国的首都是哪里？', '巴黎'), ('一周有几天？', '7'), ('水的化学式是什么？', 'H2O'), ('二进制中 1+1 等于几（十进制）？', '2')]
    for q, a in extra:
        for pref in ['', '请回答：']:
            rows.append({'id': uid('zh_eval3', pref, q), 'task_type': 'zh_brief', 'prompt': pref + q, 'gold': a, 'meta': {'fact': True}})
    return rows

def make_format_eval():
    rows = []
    rng = random.Random(SEED + 21)
    for i in range(40):
        n = rng.randint(10, 99)
        rows.append({
            'id': uid('fmt_num', n, i),
            'task_type': 'format_follow',
            'prompt': f'What is {n}+1? Reply with ONLY the number, no words.',
            'gold': str(n + 1),
            'meta': {'mode': 'number_only'},
        })
    for i in range(20):
        rows.append({
            'id': uid('fmt_json', i),
            'task_type': 'format_follow',
            'prompt': f'Return ONLY JSON: {{"ok": true, "id": {i}}} with no other text.',
            'gold': {'ok': True, 'id': i},
            'meta': {'mode': 'json_only'},
        })
    return rows

struct_train = make_struct_train(900)
json_eval, sql_eval = make_struct_eval()
zh_train = make_zh_train()
zh_eval = make_zh_eval()
format_eval = make_format_eval()
print('struct/zh/fmt', len(struct_train), len(json_eval), len(sql_eval), len(zh_train), len(zh_eval), len(format_eval))


In [ ]:
# 【步骤】可选：拉取公开指令数据切片（失败则走内置数据）

alpaca_rows = []
try:
    from datasets import load_dataset
    ds = load_dataset('tatsu-lab/alpaca', split='train[:2500]')
    for row in ds:
        instr = (row.get('instruction') or '').strip()
        inp = (row.get('input') or '').strip()
        out = (row.get('output') or '').strip()
        if not instr or not out:
            continue
        if len(out) > 800 or len(instr) > 500:
            continue
        if inp:
            instr = instr + '\n' + inp
        alpaca_rows.append({
            'id': uid('alpaca', instr[:200], out[:200]),
            'task_type': 'general',
            'instruction': instr,
            'output': out,
            'meta': {'source': 'tatsu-lab/alpaca'},
        })
    print('alpaca kept', len(alpaca_rows))
except Exception as e:
    print('alpaca skip', repr(e)[:200])


In [ ]:
# 【步骤】代码题：用子进程执行+assert，能量化「能不能跑」

eval_ids = set()
for bucket in [math_eval, code_eval, json_eval, sql_eval, zh_eval, format_eval]:
    for r in bucket:
        eval_ids.add(r['id'])
        key = normalize_text(r.get('prompt') or '')
        eval_ids.add(uid('blk', key))

train_pool = []
for bucket in [math_train, code_train, struct_train, zh_train, alpaca_rows]:
    train_pool.extend(bucket)

filtered = []
for r in train_pool:
    key = normalize_text(r['instruction'])
    if uid('blk', key) in eval_ids or r['id'] in eval_ids:
        continue
    filtered.append(r)

rng = random.Random(SEED)
rng.shuffle(filtered)
by_type = {}
for r in filtered:
    by_type.setdefault(r['task_type'], []).append(r)

selected = []
selected.extend(by_type.get('code', []))
selected.extend(by_type.get('zh', []))
for t, cap in [('math', 2000), ('json', 450), ('sql', 450), ('general', 2000)]:
    selected.extend(by_type.get(t, [])[:cap])
rng.shuffle(selected)
n_val = max(200, int(0.1 * len(selected)))  # 划分验证集：调参用，不等于最终测试
val_rows = selected[:n_val]
train_rows = selected[n_val:]
print('train', len(train_rows), 'val', len(val_rows))

def write_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

write_jsonl(DATA / 'train.jsonl', train_rows)
write_jsonl(DATA / 'val.jsonl', val_rows)
write_jsonl(EVAL / 'math_em.jsonl', math_eval)
write_jsonl(EVAL / 'code_exec.jsonl', code_eval)
write_jsonl(EVAL / 'json_valid.jsonl', json_eval)
write_jsonl(EVAL / 'sql_form.jsonl', sql_eval)
write_jsonl(EVAL / 'zh_brief.jsonl', zh_eval)
write_jsonl(EVAL / 'format_follow.jsonl', format_eval)

manifest = {
    'seed': SEED,
    'model_id_baseline': MODEL_ID,
    'train_n': len(train_rows),
    'val_n': len(val_rows),
    'eval_counts': {
        'math_em': len(math_eval),
        'code_exec': len(code_eval),
        'json_valid': len(json_eval),
        'sql_form': len(sql_eval),
        'zh_brief': len(zh_eval),
        'format_follow': len(format_eval),
    },
    'weights': {
        'math_em': 0.30,
        'code_exec': 0.30,
        'json_valid': 0.15,
        'sql_form': 0.05,
        'zh_brief': 0.10,
        'format_follow': 0.10,
    },
    'notes': 'eval_suite_v1 is frozen; never train on these ids/prompts',
}
(EVAL / 'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
(DATA / 'manifest.json').write_text(json.dumps({
    'seed': SEED,
    'train_n': len(train_rows),
    'val_n': len(val_rows),
    'train_type_hist': {k: sum(1 for r in train_rows if r['task_type'] == k) for k in sorted(set(r['task_type'] for r in train_rows))},
}, indent=2))
print(manifest)


In [ ]:
# 【步骤】代码题：用子进程执行+assert，能量化「能不能跑」

@torch.no_grad()
def generate(prompt: str, max_new: int = 128) -> str:
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new,
        do_sample=False,  # greedy：评测可复现，避免采样噪声
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def score_math(pred, gold):
    pn, gn = extract_final_number(pred), extract_final_number(str(gold))
    return float(pn is not None and gn is not None and pn == gn)

def extract_code(pred: str) -> str:
    t = pred.strip()
    m = re.search(r'```(?:python)?\s*([\s\S]*?)```', t)
    if m:
        return m.group(1).strip()
    m2 = re.search(r'(def |class )', t)
    if m2:
        return t[m2.start():].strip()
    return t

def score_code(pred, tests):
    return float(run_python_user_code(extract_code(pred), tests, timeout_s=2.0))

def score_json(pred, gold_obj):
    obj = safe_json_loads(pred)
    if not isinstance(obj, dict):
        return 0.0
    return float(all(obj.get(k) == v for k, v in gold_obj.items()))

def score_sql(pred, gold):
    p = pred.strip().strip('`')
    p = re.sub(r'^sql\s*', '', p, flags=re.I).strip()
    def norm(s):
        s = s.strip().rstrip(';').lower()
        return re.sub(r'\s+', ' ', s)
    return float(norm(p) == norm(gold))

def token_f1(a, b):
    ta, tb = normalize_text(a).split(), normalize_text(b).split()
    if not ta and not tb:
        return 1.0
    if not ta or not tb:
        return 0.0
    ca, cb = {}, {}
    for t in ta:
        ca[t] = ca.get(t, 0) + 1
    for t in tb:
        cb[t] = cb.get(t, 0) + 1
    overlap = sum(min(ca[t], cb.get(t, 0)) for t in ca)
    if overlap == 0:
        return 0.0
    prec = overlap / len(ta)
    rec = overlap / len(tb)
    return 2 * prec * rec / (prec + rec)

def score_zh(pred, gold):
    if normalize_text(pred) == normalize_text(gold):
        return 1.0
    if normalize_text(gold) in normalize_text(pred):
        return 1.0
    return float(token_f1(pred, gold) >= 0.6)

def score_format(pred, gold, mode):
    if mode == 'number_only':
        return score_math(pred, gold)
    if mode == 'json_only':
        obj = safe_json_loads(pred)
        if not isinstance(obj, dict):
            return 0.0
        return float(obj.get('ok') is True and obj.get('id') == gold.get('id'))
    return 0.0


In [ ]:
# 【步骤】注入 LoRA：只训练低秩旁路，基座权重冻结

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader

try:
    import peft.tuners.lora.torchao as peft_torchao
    peft_torchao.is_torchao_available = lambda: False
except Exception:
    pass

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, trust_remote_code=True)
base.config.use_cache = False
base.gradient_checkpointing_enable()  # 用算力换显存：激活重计算
if hasattr(base, 'enable_input_require_grads'):  # 梯度检查点时常需要，否则 LoRA 可能收不到梯度
    base.enable_input_require_grads()
cand = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
present = sorted({n.split('.')[-1] for n,_ in base.named_modules() if n.split('.')[-1] in cand})
model = get_peft_model(base, LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05, bias='none', target_modules=present))
model.to(DEVICE)

def load_jsonl(path):
    rows=[]
    with open(path, encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def format_chat(instr, resp):
    return tokenizer.apply_chat_template([{'role':'user','content':instr},{'role':'assistant','content':resp}], tokenize=False, add_generation_prompt=False)

class DS(Dataset):
    def __init__(self, rows, max_len=512):
        self.rows = [tokenizer(format_chat(r['instruction'], r['output']), truncation=True, max_length=max_len, padding=False) for r in rows]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        return self.rows[i]

def collate(b):
    return tokenizer.pad(b, padding=True, return_tensors='pt')

train = load_jsonl(DATA/'train.jsonl')
boost = [r for r in train if r['task_type'] in ('math','code','json','sql','zh')]
train = train + boost
random.Random(SEED).shuffle(train)
loader = DataLoader(DS(train), batch_size=2, shuffle=True, collate_fn=collate, drop_last=True)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
scaler = torch.amp.GradScaler('cuda', enabled=True)
STEPS = 350
model.train(); losses=[]; it=iter(loader); t0=time.perf_counter()
for step in range(STEPS):
    try:
        batch = next(it)
    except StopIteration:
        it = iter(loader); batch = next(it)
    batch = {k:v.to(DEVICE) for k,v in batch.items()}
    labels = batch['input_ids'].clone(); labels[batch['attention_mask']==0] = -100  # -100：CrossEntropy 忽略 pad（及可选 prompt）位置
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', dtype=AMP):
        loss = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=labels).loss
    scaler.scale(loss).backward(); scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)  # 梯度裁剪：防爆炸
    scaler.step(opt); scaler.update()
    losses.append(float(loss.detach().float().cpu()))
    if step % 50 == 0 or step == STEPS-1:
        print(f'step {step}/{STEPS} loss={losses[-1]:.4f}')
print('train_s', time.perf_counter()-t0)
(OUT/'adapter_release').mkdir(exist_ok=True)
model.save_pretrained(OUT/'adapter_release'); tokenizer.save_pretrained(OUT/'adapter_release')
merged = model.merge_and_unload()
merged_dir = OUT/'mini_instruct_t4_merged'
merged_dir.mkdir(exist_ok=True)
merged.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)
print('merged', merged_dir)
merged.to(DEVICE); merged.eval()


In [ ]:
# 【步骤】代码题：用子进程执行+assert，能量化「能不能跑」

@torch.no_grad()
def generate_with(m, prompt, max_new=128):
    messages=[{'role':'user','content':prompt}]
    text=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs=tokenizer(text, return_tensors='pt').to(DEVICE)
    out=m.generate(**inputs, max_new_tokens=max_new, do_sample=False,  # greedy：评测可复现，避免采样噪声
                   pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map={'':0}, trust_remote_code=True)
base.eval()
subsets = {k: load_jsonl(EVAL/f'{k}.jsonl') for k in ['math_em','code_exec','json_valid','sql_form','zh_brief','format_follow']}
weights = json.loads((EVAL/'manifest.json').read_text())['weights']

def eval_model(m, tag):
    metrics={}
    for name, rows in subsets.items():
        scores=[]
        for i,r in enumerate(rows):
            pred=generate_with(m, r['prompt'], max_new=160 if name=='code_exec' else 96)
            if name=='math_em': s=score_math(pred,r['gold'])
            elif name=='code_exec': s=score_code(pred,r['tests'])
            elif name=='json_valid': s=score_json(pred,r['gold'])
            elif name=='sql_form': s=score_sql(pred,r['gold'])
            elif name=='zh_brief': s=score_zh(pred,r['gold'])
            else: s=score_format(pred,r['gold'], r['meta'].get('mode'))
            scores.append(s)
            if (i+1)%40==0:
                print(f'{tag} {name} {i+1}/{len(rows)} avg={sum(scores)/len(scores):.3f}')
        metrics[name]=sum(scores)/len(scores)
        print(f'{tag} == {name}: {metrics[name]:.4f}')
    w=sum(metrics[k]*weights[k] for k in weights)
    print(tag,'WEIGHTED',w)
    return metrics,w

print('BASE'); bm,bw=eval_model(base,'base')
print('MERGED'); mm,mw=eval_model(merged,'merged')
rel=(mw-bw)/max(1e-8,bw)
print('relative_gain', rel)


In [ ]:
# 【步骤】统一评测：固定 greedy 解码 + 加权主分

card = '''# Mini-Instruct-T4 Model Card

## Summary
Short-form instruct model for math, Python snippets, JSON/SQL, and brief ZH/EN answers.
LoRA-tuned from Qwen2.5-0.5B-Instruct (SEED=42) then merged.

## Metrics (greedy)
- base weighted: %.4f
- merged weighted: %.4f
- relative gain: %.2f%%

Base metrics: %s
Merged metrics: %s

## Load
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
m = AutoModelForCausalLM.from_pretrained('mini_instruct_t4_merged', torch_dtype='auto', device_map='auto')
t = AutoTokenizer.from_pretrained('mini_instruct_t4_merged')
```

## Limits
0.5B ceiling; synthetic-heavy data; not a general chatbot.
''' % (bw, mw, rel*100, json.dumps(bm), json.dumps(mm))
(OUT/'MODEL_CARD.md').write_text(card)
final = {
    'notebook': 'grok-007-release-merge-card',
    'phase': 'M3_release',
    'model_id': MODEL_ID,
    'merged_path': str(merged_dir),
    'adapter_path': str(OUT/'adapter_release'),
    'baseline_weighted': bw,
    'final_weighted': mw,
    'relative_gain': rel,
    'baseline_metrics': bm,
    'final_metrics': mm,
    'weights': weights,
    'gate_15pct': rel >= 0.15,
    'model_card': str(OUT/'MODEL_CARD.md'),
}
(OUT/'final_report.json').write_text(json.dumps(final, indent=2, ensure_ascii=False))
(OUT/'grok007_results.json').write_text(json.dumps(final, indent=2, ensure_ascii=False))
print(json.dumps({k: final[k] for k in ['baseline_weighted','final_weighted','relative_gain','gate_15pct']}, indent=2))
print('DONE M3')


## 学习检查清单

- 你应能回答：merge 前后产物差异？模型卡至少写清哪些字段？
- 建议：改一个超参重跑一小段，观察 log 变化（比只读代码更有效）。
